### __Predicting Airbnb Listing Prices in Sydney__

--- 
## Task 3: Feature Engineering


### Steps to select the best features from the dataset

Feature Engineering is essential to improve regression model performance. This task focuses on feature selection, which helps in reducing dimensionality and improving model performance by keeping only the most relevant features. Since some features have low correlation or importance, we should refine our feature selection process. Here’s the approach we should take:

**1️. Use Only Highly Correlated Features**
- Select features that have a strong correlation (e.g., `|correlation| > 0.2`) with `price`.
- Drop weakly correlated features (like most of the date-based ones).

**2. Remove Multicollinearity**
- If two features are highly correlated with each other (e.g., `bedrooms` and `beds`), keep only one.
- This prevents redundant information in the model.

In [1]:
# Import required libraries
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_regression

In [2]:
# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Configure seaborn aesthetics
sns.set(style="whitegrid", palette="muted", font_scale=1.2)

Firstly, we import the processed data from the completed data cleaning step: 

In [3]:
# Import both the training and testing data
#df_train = pd.read_csv(r"C:\Users\haiho\GITHUB\Sydney-Airbnb-prices-prediction\data\processed\processed_train.csv")
#df_test = pd.read_csv(r"C:\Users\haiho\GITHUB\Sydney-Airbnb-prices-prediction\data\processed\processed_test.csv")

In [9]:
df_train = pd.read_csv("C:\DVS_hoangnh272\Material\Github\Sydney-Airbnb-prices-prediction\data\processed\processed_train.csv")
df_test = pd.read_csv("C:\DVS_hoangnh272\Material\Github\Sydney-Airbnb-prices-prediction\data\processed\processed_test.csv")

In [10]:
# Define a list of DataFrames for repeated uses
dfs = [df_train, df_test]

### 1. Use Only Highly Correlated Features

To explore and measure the relationship between current features and `price` as the target variable, we can construct correlation matrix to establish them.

There still exists some text-based columns which are too general and irrelevant for predict price, we will drop them before executing the correlation matrix. 

In [11]:
# Exclude object columns
drop_cols = ["host_name", "host_location", "host_neighbourhood", "host_since", "neighbourhood"]
df_train.drop(columns=drop_cols, inplace=True, errors="ignore")

In [12]:
# Correlation Analysis
correlation_matrix = df_train.corr()
price_correlations = correlation_matrix["price"].sort_values(ascending=False)
price_correlations

price                   1.000000
price_per_bedroom       0.901433
bedrooms                0.654258
accommodates            0.597608
beds                    0.531221
                          ...   
reviews_per_month      -0.112386
response_time          -0.113732
number_of_reviews      -0.117206
host_acceptance_rate   -0.151383
mapped_bathrooms       -0.304709
Name: price, Length: 55, dtype: float64

Features with **very low correlation (< 0.2)** contribute little to linear models and may introduce noise. And if we set the threshold **too high (e.g., 0.5 or 0.7)**, we may miss important features that contribute slightly but collectively improve the model. Therefore, based on the correlation matrix output, a threshold of 0.2 is suitable.

In [13]:
# Define a correlation threshold
corr_threshold = 0.2

In [14]:
# Select features with correlation > threshold (excluding 'price' itself)
selected_features_corr = price_correlations[abs(price_correlations) > corr_threshold].index.tolist()
selected_features_corr.remove("price")  # Remove target variable
selected_features_corr

['price_per_bedroom',
 'bedrooms',
 'accommodates',
 'beds',
 'mapped_property_type',
 'mapped_room_type',
 'mapped_bathrooms']

### 2. Remove Features with Mutual Information

In current situation of both datasets, there exist features with significant mutual information that will cause redundancy and irrelevance: 
- `neighbourhood_cleansed` and `neighbourhood` likely contain similar information.
- `host_location` may not be necessary if it's the same as `neighbourhood`.

In [15]:
# Create subsets of features
X = df_train.drop(columns=["price"])
y = df_train["price"]

In [18]:
print(X.isnull().sum())  # Total number of NaNs in X


host_response_rate         0
host_acceptance_rate       0
host_is_superhost          0
host_listings_count        0
host_has_profile_pic       0
                        ... 
mapped_property_type       0
mapped_bathrooms           0
mapped_room_type         134
response_time            645
neighbourhood_encoded      0
Length: 54, dtype: int64


In [ ]:
# Compute mutual information scores for feature selection
mutual_info = mutual_info_regression(X, y)
mutual_info_series = pd.Series(mutual_info, index=X.columns)

As Mutual information (MI) captures **non-linear relationships** unlike correlation, MI values are often much **smaller than correlation values**, so a lower threshold is reasonable.

Even a small MI value of 0.01 can indicate that a feature has some useful contribution.

In [ ]:
# Set a reasonable threshold
mi_threshold = 0.01  

In [ ]:
# Filter features with significant mutual information
selected_features_mi = mutual_info_series[mutual_info_series > mi_threshold].index.tolist()
selected_features_mi

Final feature selection: keep features that pass either correlation or mutual info filtering

In [ ]:
# Final list of selected features (intersection of correlation and mutual information)
final_selected_features = list(set(selected_features_corr) & set(selected_features_mi))

In [ ]:
selected_features.append("price")  # Ensure the target variable is kept

In [ ]:
# Drop weak features from the dataset
df_selected = df[final_selected_features + ["price"]]

### 3. Save Feature-Engineered Data

In [ ]:
# Save the updated dataset with feature-engineered columns
feature_engineered_file_path = "/mnt/data/feature_engineered_train.csv"
df_selected.to_csv(feature_engineered_file_path, index=False)
